In [79]:
import psycopg2
from dotenv import load_dotenv
import os

In [80]:
# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Connect to the database
try:
    connection = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )
    print("Connection successful!")
    
    # Create a cursor to execute SQL queries
    cursor = connection.cursor()
    
    # Example query
    cursor.execute("SELECT NOW();")
    result = cursor.fetchone()
    print("Current Time:", result)

    # Close the cursor and connection
    cursor.close()
    connection.close()
    print("Connection closed.")

except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!
Current Time: (datetime.datetime(2026, 3, 13, 3, 22, 27, 112310, tzinfo=datetime.timezone.utc),)
Connection closed.


In [81]:
from google import genai
import pandas as pd

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
GEMINI_MODEL = os.getenv("GEMINI_MODEL_CODE", "gemini-2.5-flash")
print("Gemini listo:", GEMINI_MODEL)

Gemini listo: gemini-2.5-flash


In [82]:
import json
from psycopg2.extras import execute_values

SCHEMA_CONTEXT = """
Tabla: haciendas
Granularidad: un registro = una hacienda + un mes.
Cobertura temporal: enero 2020 a junio 2025.

Columnas:
- FECHA (date): fecha del registro (tipo DATE en PostgreSQL, formato YYYY-MM-DD)
- Semana (int): número de semana del año
- Zona (text): región geográfica
- Unidad (text): código identificador de la hacienda
- Nombre_Unidad (text): nombre de la hacienda
- Real (numeric): indicador de rendimiento/ratio de producción real
- Costo_Ha (numeric): costo total acumulado por hectárea
- Atencion_Plantacion (numeric): costos de mantenimiento del cultivo
- C_Riego (numeric): costo total de riego
- C_Mano_Obra_Riego (numeric): costo de personal para riego
- C_Mantenimiento_Riego (numeric): costo de reparaciones de infraestructura de riego
- C_Combustible (numeric): costo de combustible
- C_Control_Sigatoca (numeric): costo del programa contra la Sigatoka
- C_Aplicacion_Aerea (numeric): costo de fumigación aérea
- C_Deshoje (numeric): costo de deshoje
- C_Costos_Productos (numeric): costo en insumos químicos y fertilizantes
- C_Fertilizacion (numeric): costo total de fertilización
- C_Sacos_Fert (numeric): costo de sacos de fertilizante
- C_ManodeObra_Fert (numeric): costo de aplicación de fertilizante
- C_Transporte_Fert (numeric): costo de transporte de fertilizante
- C_Administracion_Hacienda (numeric): costos administrativos
- Sueldos (numeric): nómina de empleados fijos
- Servicios_Basicos (numeric): pagos de luz, agua y otros servicios
- C_Empaque_Fijo (numeric): costos fijos de empaque
- Mantenimiento_Empacadora (numeric): costos de mantenimiento de empacadora
- Mantenimiento_Equipo (numeric): costo de mantenimiento de equipo
- C_Logistica (numeric): costo total de logística
- Transporte (numeric): gasto en fletes y acarreo
- Materiales (numeric): inversión en insumos de empaque
- Reclasificaciones_Transporte (numeric): ajustes contables sobre transporte
- Reclasificaciones_Materiales (numeric): ajustes contables sobre materiales
- C_Empaque_Variable (numeric): costos variables de empaque
- C_Cosecha (numeric): costo de cosecha
- C_Transporte (numeric): otros costos de transporte
- C_Depreciaciones (numeric): depreciación de activos fijos
- Total_Cajas (numeric): volumen total de cajas producidas
- Total_Hectareas (numeric): superficie productiva en hectáreas
- Racimo_Rechazado (numeric): cantidad de fruta rechazada
- Total_Peso_Caja (numeric): peso total de las cajas
- Promedio_Peso_Caja (numeric): peso promedio por caja
- Tipo_Suelo (text): clasificación del terreno
- Incidencia_Sigatoka (numeric): nivel de presencia de la plaga
- Temperatura_C (numeric): temperatura media en °C
- Precipitacion_mm (numeric): lluvia acumulada en mm
- Evotranspiracion (numeric): tasa de evapotranspiración
- Humedad (numeric): porcentaje de humedad relativa
- Ausentismo_Agricola (numeric): total de inasistencias del personal
- Ausentismo_Justificado_Agricola (numeric): inasistencias justificadas
- Ausentismo_Injustificado_Agricola (numeric): inasistencias injustificadas
- RotPerson_Salida_Todos_Motivos_Agricola (numeric): índice de rotación de personal
- Pago_Labor_Persona (numeric): indicador de pago por jornada
- Pago_Por_Cuenta (numeric): indicador de pagos por cuenta
- Vacante_Labor (numeric): puestos de trabajo vacantes
"""

COLUMNAS_VALIDAS = [
    line.split("(")[0].replace("-", "").strip()
    for line in SCHEMA_CONTEXT.splitlines()
    if line.strip().startswith("-")
]
COLUMNAS_VALIDAS_STR = ", ".join(f'"{c}"' for c in COLUMNAS_VALIDAS)

print(f"Schema cargado — {len(COLUMNAS_VALIDAS)} columnas")

Schema cargado — 53 columnas


In [83]:
def _llm(prompt: str) -> str:
    r = client.models.generate_content(
        model=GEMINI_MODEL, contents=prompt, config={"temperature": 0.0}
    )
    return r.text.strip()


def _limpiar_sql(texto: str) -> str:
    texto = texto.strip()
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("sql"):
            texto = texto[3:]
    return texto.strip()


def _conectar():
    return psycopg2.connect(
        user=os.getenv("user"), password=os.getenv("password"),
        host=os.getenv("host"), port=os.getenv("port"), dbname=os.getenv("dbname")
    )

print("Helpers listos")

Helpers listos


In [84]:
def hacer_plan(pregunta: str) -> list[str]:
    """Devuelve una lista de pasos donde:
    - Paso 1: extracción de datos de la BD con SQL
    - Pasos 2+: operaciones sobre el DataFrame de pandas resultante
    """
    prompt = f"""Eres un experto en análisis de datos con SQL y pandas para Python.

Tienes la tabla "haciendas" en PostgreSQL con las columnas:
{COLUMNAS_VALIDAS_STR}

El usuario pregunta: "{pregunta}"

Genera un plan de resolución con estas reglas:
- El PRIMER paso debe describir qué datos extraer de la base de datos con SQL (una sola consulta amplia).
- Los pasos SIGUIENTES deben describir operaciones de transformación, filtrado o cálculo sobre el DataFrame de pandas resultante del paso anterior. NO deben consultar la base de datos.

Responde ÚNICAMENTE con un JSON array de strings, sin markdown. Ejemplo:
[
  "Extraer de la BD: Nombre_Unidad, Costo_Ha, Total_Cajas, FECHA para el año 2024",
  "Calcular el costo por caja dividiendo Costo_Ha entre Total_Cajas",
  "Ordenar de mayor a menor por costo por caja y quedarse con las 5 primeras"
]"""

    texto = _llm(prompt)
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    return json.loads(texto.strip().rstrip("```").strip())

In [85]:
def generar_sql(actividad: str, pregunta_original: str) -> tuple[str, str]:
    """Paso 1: el LLM razona y genera el SQL de extracción contra la BD."""
    prompt = f"""Eres un experto en SQL para PostgreSQL y en el negocio bananero.

Tabla disponible:
{SCHEMA_CONTEXT}

Pregunta original del usuario: "{pregunta_original}"

Genera el SQL para extraer los datos necesarios para esta actividad:
"{actividad}"

Responde en este formato exacto:

RAZONAMIENTO:
<explica qué columnas extraes y por qué, en 2-4 oraciones>

SQL:
<solo el SQL, con saltos de línea e indentación>

Reglas del SQL:
- Tabla: "haciendas"
- Extrae columnas suficientes para que los pasos siguientes puedan operar sobre el DataFrame sin volver a la BD
- FECHA es DATE; para filtrar/agrupar por año: EXTRACT(YEAR FROM "FECHA")
- Comillas dobles en TODOS los nombres de columna
- Solo estas columnas (PROHIBIDO inventar otras): {COLUMNAS_VALIDAS_STR}
- Sin markdown, sin backticks"""

    texto = _llm(prompt)
    razonamiento, sql = "", ""
    if "SQL:" in texto:
        partes = texto.split("SQL:", 1)
        razonamiento = partes[0].replace("RAZONAMIENTO:", "").strip()
        sql = _limpiar_sql(partes[1])
    else:
        sql = _limpiar_sql(texto)
    return razonamiento, sql

In [86]:
def corregir_sql(sql: str) -> str:
    """Revisa el SQL y corrige cualquier nombre de columna que no exista en la tabla haciendas."""
    prompt = f"""Eres un experto en SQL para PostgreSQL.

Tienes la tabla "haciendas" con EXACTAMENTE estas columnas (respeta mayúsculas/minúsculas):
{COLUMNAS_VALIDAS_STR}

Revisa el siguiente SQL y corrige únicamente los nombres de columna que no correspondan a la lista anterior.
Si un nombre de columna no existe, reemplázalo por el nombre correcto más cercano de la lista.
No cambies nada más (lógica, filtros, alias de resultado, etc.).
Si el SQL ya es correcto, devuélvelo tal cual.

SQL a revisar:
{sql}

Devuelve SOLO el SQL corregido, sin explicaciones, sin markdown, sin backticks."""

    return _limpiar_sql(_llm(prompt))

In [87]:
def generar_pandas(actividad: str, pregunta_original: str, df: pd.DataFrame) -> tuple[str, str]:
    """Pasos 2+: el LLM razona y genera código pandas que opera sobre el DataFrame actual."""
    info_df = (
        f"Columnas: {list(df.columns)}\n"
        f"Tipos:    {df.dtypes.to_dict()}\n"
        f"Filas:    {len(df)}\n"
        f"Muestra:\n{df.head(5).to_string(index=False)}"
    )

    prompt = f"""Eres un experto en análisis de datos con pandas para Python.

Tienes un DataFrame llamado `df` con esta estructura:
{info_df}

Pregunta original del usuario: "{pregunta_original}"

Genera el código pandas para realizar esta operación sobre `df`:
"{actividad}"

Responde en este formato exacto:

RAZONAMIENTO:
<explica qué operación realizas y por qué, en 2-4 oraciones>

CODIGO:
<solo el código Python/pandas>

Reglas del código:
- El DataFrame de entrada se llama `df`
- El resultado final SIEMPRE debe quedar guardado en `df` como un pandas DataFrame (NO Series, NO lista, NO dict)
- Si usas .groupby(), termina con .reset_index() para que el resultado sea un DataFrame
- Si usas .value_counts() o cualquier método que devuelva una Series, conviértela a DataFrame con .reset_index()
- Solo usa pandas, no importes librerías adicionales
- Sin markdown, sin backticks"""

    texto = _llm(prompt)
    razonamiento, codigo = "", ""
    if "CODIGO:" in texto:
        partes = texto.split("CODIGO:", 1)
        razonamiento = partes[0].replace("RAZONAMIENTO:", "").strip()
        codigo = partes[1].strip()
        if codigo.startswith("```"):
            codigo = codigo.split("```")[1]
            if codigo.startswith("python"):
                codigo = codigo[6:]
        codigo = codigo.strip()
    else:
        codigo = texto.strip()
    return razonamiento, codigo

In [88]:
def ejecutar_pipeline(pregunta: str) -> pd.DataFrame | None:
    print(f"Pregunta: {pregunta}")
    print("=" * 60)

    pasos = hacer_plan(pregunta)
    print(f"Plan ({len(pasos)} actividades):")
    for i, p in enumerate(pasos, 1):
        print(f"  {i}. {p}")
    print()

    df = None

    # ── Paso 1: extracción SQL desde la BD ───────────────────
    print(f"─── Paso 1 (SQL): {pasos[0]}")
    try:
        razonamiento, sql = generar_sql(pasos[0], pregunta)
        print(f"Razonamiento:\n{razonamiento}\n")
        print(f"SQL generado:\n{sql}\n")

        sql_ok = corregir_sql(sql)
        if sql_ok != sql:
            print(f"SQL corregido:\n{sql_ok}\n")

        conn = _conectar()
        try:
            cur = conn.cursor()
            cur.execute(sql_ok)
            cols = [desc[0] for desc in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=cols)
            cur.close()
        finally:
            conn.close()

        print(f"DataFrame extraído: {len(df)} filas × {len(df.columns)} columnas")
        display(df.head())
    except Exception as e:
        print(f"ERROR en paso 1: {e}")
        return None
    print()

    # ── Pasos 2+: operaciones pandas sobre el DataFrame ──────
    for i, actividad in enumerate(pasos[1:], 2):
        print(f"─── Paso {i} (pandas): {actividad}")
        try:
            razonamiento, codigo = generar_pandas(actividad, pregunta, df)
            print(f"Razonamiento:\n{razonamiento}\n")
            print(f"Código:\n{codigo}\n")

            local_ns = {"df": df, "pd": pd}
            exec(codigo, {}, local_ns)
            resultado = local_ns["df"]

            # guard: garantizar que siempre sea DataFrame
            if isinstance(resultado, pd.Series):
                resultado = resultado.reset_index()

            df = resultado
            print(f"Resultado: {len(df)} filas × {len(df.columns)} columnas")
            display(df)
        except Exception as e:
            print(f"ERROR en paso {i}: {e}")
        print()

    return df


"Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?"


Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?
¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?
¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?
¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?
¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?
¿Qué variables administrativas, pueden estar afectando los costos?
¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?
¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?
¿Qué programa se debería de ajustar para mejorar la productividad?

In [89]:
pregunta = "¿Cuáles son las 5 haciendas con mayor costo por hectárea en 2024?"

df_resultado = ejecutar_pipeline(pregunta)

Pregunta: ¿Cuáles son las 5 haciendas con mayor costo por hectárea en 2024?
Plan (3 actividades):
  1. Extraer de la BD: Nombre_Unidad, Costo_Ha, FECHA de la tabla haciendas donde el año de FECHA sea 2024.
  2. Agrupar el DataFrame por 'Nombre_Unidad' y calcular el promedio de 'Costo_Ha' para cada hacienda.
  3. Ordenar el DataFrame resultante de mayor a menor por el promedio de 'Costo_Ha' y seleccionar las 5 primeras haciendas.

─── Paso 1 (SQL): Extraer de la BD: Nombre_Unidad, Costo_Ha, FECHA de la tabla haciendas donde el año de FECHA sea 2024.
Razonamiento:
Se extraen las columnas "Nombre_Unidad" para identificar cada hacienda, "Costo_Ha" para analizar el costo por hectárea, y "FECHA" para poder filtrar los registros correspondientes al año 2024, tal como lo solicita la actividad. Estas columnas son esenciales para el análisis posterior de los costos.

SQL generado:
SELECT
    "Nombre_Unidad",
    "Costo_Ha",
    "FECHA"
FROM
    haciendas
WHERE
    EXTRACT(YEAR FROM "FECHA") = 20

,Nombre_Unidad,Costo_Ha,FECHA
0,Hacienda San Escriva,1129.323932,2024-01-01
1,Hacienda San Escriva,1129.110624,2024-02-01
2,Hacienda San Escriva,1073.093559,2024-03-01
3,Hacienda San Escriva,1143.437815,2024-04-01
4,Hacienda San Escriva,1324.850388,2024-05-01



─── Paso 2 (pandas): Agrupar el DataFrame por 'Nombre_Unidad' y calcular el promedio de 'Costo_Ha' para cada hacienda.
Razonamiento:
Para encontrar el costo promedio por hectárea para cada hacienda, primero agrupamos el DataFrame por la columna 'Nombre_Unidad'. Luego, calculamos la media de la columna 'Costo_Ha' para cada uno de estos grupos. Finalmente, usamos reset_index() para convertir el resultado, que inicialmente sería una Serie, de nuevo en un DataFrame con 'Nombre_Unidad' como una columna regular.

Código:
df = df.groupby('Nombre_Unidad')['Costo_Ha'].mean().reset_index()

Resultado: 44 filas × 2 columnas


,Nombre_Unidad,Costo_Ha
0,Hacienda Admiracion,1260.015608
1,Hacienda Agrilechos I,973.291564
2,Hacienda Agrilechos II,1018.219059
3,Hacienda Banalandia,1145.382221
4,Hacienda Banalandia II,1214.768395
5,Hacienda Calope,1588.092423
6,Hacienda Chollo,1090.459302
7,Hacienda Continental,1036.962267
8,Hacienda Cristal,1048.752673
9,Hacienda Cristal II,1045.089068



─── Paso 3 (pandas): Ordenar el DataFrame resultante de mayor a menor por el promedio de 'Costo_Ha' y seleccionar las 5 primeras haciendas.
Razonamiento:
Para encontrar las 5 haciendas con mayor costo por hectárea, primero agrupamos el DataFrame por 'Nombre_Unidad' y calculamos el promedio de 'Costo_Ha' para cada hacienda. Luego, ordenamos este resultado de mayor a menor por el costo promedio y seleccionamos las 5 primeras entradas.

Código:
df = df.groupby('Nombre_Unidad')['Costo_Ha'].mean().reset_index().sort_values(by='Costo_Ha', ascending=False).head(5)

Resultado: 5 filas × 2 columnas


,Nombre_Unidad,Costo_Ha
38,Hacienda Union I,1848.244545
31,Hacienda San Carlos,1613.309074
5,Hacienda Calope,1588.092423
17,Hacienda Maravilla III,1284.517886
0,Hacienda Admiracion,1260.015608


Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?
¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?
¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?
¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?
¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?
¿Qué variables administrativas, pueden estar afectando los costos?
¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?
¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?
¿Qué programa se debería de ajustar para mejorar la productividad?

In [ ]:
preguntas = [
    "Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?",
    "Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?",
    "¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?",
    "¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?",
    "¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?",
    "¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?",
    "¿Qué variables administrativas, pueden estar afectando los costos?",      
    "¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?",  
    "¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?",
    "¿Qué programa se debería de ajustar para mejorar la productividad?",

]

for p in preguntas:
    df_resultado = ejecutar_pipeline(p)

Pregunta: Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Plan (7 actividades):
  1. Extraer de la BD: FECHA, Nombre_Unidad, Real, Total_Cajas, Total_Hectareas de la tabla haciendas.
  2. Calcular una nueva columna 'Costo_Total_Estimado' para cada registro, multiplicando la columna 'Real' (interpretada como el costo por hectárea bajo cumplimiento del 100% de programas) por 'Total_Hectareas'.
  3. Sumar el 'Costo_Total_Estimado' de todos los registros para obtener el costo total estimado global.
  4. Sumar 'Total_Cajas' de todos los registros para obtener el total de cajas global.
  5. Sumar 'Total_Hectareas' de todos los registros para obtener el total de hectáreas global.
  6. Calcular el 'Costo_Caja_Real' dividiendo la suma total de 'Costo_Total_Estimado' entre la suma total de 'Total_Cajas', asegurando manejar posibles divisiones por cero.
  7. Calcular el 'Costo_Ha_Real' dividiendo la suma total de '

,FECHA,Nombre_Unidad,Real,Total_Cajas,Total_Hectareas
0,2020-01-01,Hacienda San Escriva,5.114322,45513.5596,168.77
1,2020-02-01,Hacienda San Escriva,6.033120,37938.4164,168.77
2,2020-03-01,Hacienda San Escriva,4.495248,47021.6505,168.77
3,2020-04-01,Hacienda San Escriva,6.130292,40529.2263,168.77
4,2020-05-01,Hacienda San Escriva,6.329382,36093.5717,168.77



─── Paso 2 (pandas): Calcular una nueva columna 'Costo_Total_Estimado' para cada registro, multiplicando la columna 'Real' (interpretada como el costo por hectárea bajo cumplimiento del 100% de programas) por 'Total_Hectareas'.
Razonamiento:
Para calcular el 'Costo_Total_Estimado' para cada registro, se multiplican los valores de la columna 'Real' (que representa el costo por hectárea bajo cumplimiento del 100% de programas) por los valores correspondientes de la columna 'Total_Hectareas'. Esta operación se realiza de forma directa y elemento a elemento entre las dos columnas.

Código:
df['Costo_Total_Estimado'] = df['Real'] * df['Total_Hectareas']

Resultado: 2904 filas × 6 columnas


,FECHA,Nombre_Unidad,Real,Total_Cajas,Total_Hectareas,Costo_Total_Estimado
0,2020-01-01,Hacienda San Escriva,5.114322,45513.5596,168.77,863.144127
1,2020-02-01,Hacienda San Escriva,6.033120,37938.4164,168.77,1018.209579
2,2020-03-01,Hacienda San Escriva,4.495248,47021.6505,168.77,758.663076
3,2020-04-01,Hacienda San Escriva,6.130292,40529.2263,168.77,1034.609415
4,2020-05-01,Hacienda San Escriva,6.329382,36093.5717,168.77,1068.209786
...,...,...,...,...,...,...
2899,2025-02-01,Hacienda Recreo,NaN,16894.4526,79.77,NaN
2900,2025-03-01,Hacienda Recreo,NaN,16735.9286,79.77,NaN
2901,2025-04-01,Hacienda Recreo,NaN,18613.4287,79.77,NaN
2902,2025-05-01,Hacienda Recreo,NaN,17773.7868,79.77,NaN



─── Paso 3 (pandas): Sumar el 'Costo_Total_Estimado' de todos los registros para obtener el costo total estimado global.
Razonamiento:
Para obtener el costo total estimado global, se realiza una suma directa de todos los valores presentes en la columna 'Costo_Total_Estimado' de todo el DataFrame. El resultado, que es un único valor escalar, se encapsula en un nuevo DataFrame con una columna descriptiva para cumplir con el formato de salida requerido.

Código:
costo_total_estimado_global = df['Costo_Total_Estimado'].sum()
df = pd.DataFrame({'Costo_Total_Estimado_Global': [costo_total_estimado_global]})

Resultado: 1 filas × 1 columnas


...


─── Paso 5 (pandas): Calcular la matriz de correlación entre 'Productividad', 'Promedio_Peso_Caja', 'Tasa_Merma' y todas las demás columnas numéricas relevantes (costos, ambientales, laborales) para identificar los factores con mayor impacto.
Razonamiento:
Calculamos la matriz de correlación para todas las columnas numéricas presentes en el DataFrame. Esto nos permite identificar las relaciones lineales entre los indicadores de producción (productividad, peso, merma) y cualquier otro factor numérico (como costos, variables ambientales o laborales, si estuvieran presentes en el DataFrame), revelando cuáles tienen el mayor impacto o asociación.

Código:
numerical_columns = df.select_dtypes(include='number').columns
df = df[numerical_columns].corr()

Resultado: 3 filas × 3 columnas


,Productividad,Promedio_Peso_Caja,Tasa_Merma
Productividad,1.000000,-0.359601,-0.098020
Promedio_Peso_Caja,-0.359601,1.000000,-0.199142
Tasa_Merma,-0.098020,-0.199142,1.000000



─── Paso 6 (pandas): Analizar la variación de 'Productividad', 'Promedio_Peso_Caja' y 'Tasa_Merma' por 'Tipo_Suelo', 'Zona' y 'Nombre_Unidad' para identificar diferencias significativas entre categorías.
Razonamiento:
La operación solicitada de analizar la variación de 'Productividad', 'Promedio_Peso_Caja' y 'Tasa_Merma' por 'Tipo_Suelo', 'Zona' y 'Nombre_Unidad' no puede realizarse directamente con el DataFrame `df` proporcionado. El DataFrame `df` solo contiene las columnas de indicadores de producción y carece de las columnas categóricas 'Tipo_Suelo', 'Zona' y 'Nombre_Unidad' necesarias para la agrupación. El código a continuación muestra cómo se realizaría este análisis si las columnas estuvieran presentes. Sin embargo, al ejecutarlo con el `df` actual, resultaría en un KeyError, lo que impide que el resultado final sea guardado en `df` como un DataFrame.

Código:
df = df.groupby(['Tipo_Suelo', 'Zona', 'Nombre_Unidad'])[['Productividad', 'Promedio_Peso_Caja', 'Tasa_Merma']].mean()

,index,Productividad,Promedio_Peso_Caja,Tasa_Merma
0,Nombre_Unidad,-0.098020,-0.199142,1.000000
1,Tipo_Suelo,1.000000,-0.359601,-0.098020
2,Zona,-0.359601,1.000000,-0.199142



─── Paso 7 (pandas): Identificar los 3-5 factores (costos, ambientales, laborales, etc.) que muestran la correlación más fuerte o la variación más significativa con las tendencias observadas en 'Productividad', 'Promedio_Peso_Caja' y 'Tasa_Merma', y atribuir la tendencia a estos factores.
